### Building a GPT

In [22]:
import tiktoken
import torch

In [23]:
import os

path = 'C:\ptoh\DeepLLM\data\input.txt'
if os.path.exists(path):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
else:
    text = ''
    print(f"No file found at {path}. Using empty fallback text.")

<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Abhishek Raj\AppData\Local\Temp\ipykernel_28540\3289695073.py:3: SyntaxWarning: invalid escape sequence '\p'
  path = 'C:\ptoh\DeepLLM\data\input.txt'


In [24]:
print(f"length of dataset in characters : {len(text)}")
print(text[:100])

length of dataset in characters : 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [25]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [26]:
# creating mappings from chars to integers
stoi  = {ch:i for i , ch in enumerate(chars)}
itos = {i:ch for i , ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]   #takes a string outputs a list of integers 
decode = lambda l: "".join(itos[i] for i in l) #takes a list of integers and outputs a string

print(encode("hi there"))
print(decode([46, 47, 1, 58, 46, 43, 56, 43]))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [27]:
data = torch.tensor(encode(text) , dtype = torch.long)
print(data.shape,data.dtype)
print(data[:10])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [28]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [29]:
# the maximum length of chucks fed into the transformer is the block_size or context length
block_size = 8
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

in the context of 18 47 comes next in the context of 18,47 56 comes next

In [30]:
x = train_data[:block_size]
y= train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input : {context} the target : {target}")

when input : tensor([18]) the target : 47
when input : tensor([18, 47]) the target : 56
when input : tensor([18, 47, 56]) the target : 57
when input : tensor([18, 47, 56, 57]) the target : 58
when input : tensor([18, 47, 56, 57, 58]) the target : 1
when input : tensor([18, 47, 56, 57, 58,  1]) the target : 15
when input : tensor([18, 47, 56, 57, 58,  1, 15]) the target : 47
when input : tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target : 58


Now we introduce a batch dimension

In [41]:
torch.manual_seed(1337)
batch_size = 4  #how many independent sequence we process in parallel in forward backward pass
block_size = 8  #maximum context length for prediction.

def get_batch(split):
    # generate a small batches of data of input x and target y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size , (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size +1] for i in ix])
    return x , y 

xb , yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('--'*5)

for b in range(batch_size):       #batch dimension 
    for t in range(block_size):   #time dimension
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target :{target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----------
when input is [24] the target :43
when input is [24, 43] the target :58
when input is [24, 43, 58] the target :5
when input is [24, 43, 58, 5] the target :57
when input is [24, 43, 58, 5, 57] the target :1
when input is [24, 43, 58, 5, 57, 1] the target :46
when input is [24, 43, 58, 5, 57, 1, 46] the target :43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [44] the target :53
when input is [44, 53] the target :56
when input is [44, 53, 56] the target :1
when input is [44, 53, 56, 1] the target :58
when input is [44, 53, 56, 1, 58] the target :46
when input is [

### Working on the simplest Model Bi-Gram model

In [58]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)


class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        #each token directly reads off the logits for the next token from the lookup table
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self,idx,targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  # (B,T,C)   C is the possible next characters
        # as cross_entropy takes in logits in format(B,C,T)
        
        if targets is None:
            loss = None
        else:    
            B ,T ,C = logits.shape
            logits = logits.view(B*T , C)  #flatten out B and T
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits ,targets) 
        return logits , loss 
    
    def generate(self,idx ,max_new_tokens):
        # idx is (B,T) array of indices in the current context 
        for _ in range(max_new_tokens):
            #get the predictions
            logits , loss = self(idx)  #this go to the forward() of self 
            logits = logits[:,-1,:] # becomes (B , C)
            # apply softmax to get probabilities
            probs = F.softmax(logits , dim=-1) # (B,C)
            # sample from the distribution
            idx_next = torch.multinomial(probs,num_samples=1)
            #append sampled index to the running sequence
            idx = torch.cat((idx,idx_next) , dim=1) # ( B ,T+1)
        return idx    

      

In [59]:
m = BigramLanguageModel(vocab_size)
logits,loss = m(xb , yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [60]:
print(decode(m.generate(idx=torch.zeros((1,1),dtype = torch.long),max_new_tokens=100)[0].tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


Creating the optimizer

In [61]:
optimizer = torch.optim.AdamW(m.parameters() ,lr = 1e-3)

In [108]:
batch_size = 32
for steps in range(100):
    
    xb , yb = get_batch('train')
    # sample a batch of data
    logits , loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    print(loss.item())

2.648611307144165
2.6537845134735107
2.6272740364074707
2.71122407913208
2.6157758235931396
2.62644100189209
2.594118118286133
2.6991512775421143
2.6379666328430176
2.675678253173828
2.5845165252685547
2.5809409618377686
2.570181369781494
2.811061143875122
2.7244091033935547
2.5766165256500244
2.7015411853790283
2.6284260749816895
2.63291597366333
2.6437642574310303
2.574622392654419
2.7148046493530273
2.752563953399658
2.658461093902588
2.6810951232910156
2.6688809394836426
2.6595377922058105
2.6385490894317627
2.69891619682312
2.6151885986328125
2.677095890045166
2.5933687686920166
2.672147035598755
2.7163808345794678
2.54526424407959
2.672844409942627
2.806971311569214
2.7046217918395996
2.8460655212402344
2.5143020153045654
2.655235528945923
2.5854945182800293
2.621670722961426
2.6161892414093018
2.669869899749756
2.6299471855163574
2.7572848796844482
2.7377517223358154
2.6564340591430664
2.6132965087890625
2.6138579845428467
2.697479486465454
2.728119134902954
2.6777985095977783
2

In [110]:
print(decode(m.generate(idx = torch.zeros((1,1),dtype = torch.long), max_new_tokens=500)[0].tolist()))


R:
Wing ceereAcmesi s bh'uEHAue R:?Y!
F f bls w onevetrin be;

WPg the.
R
,
BUzrng
CEQ-eyg-xqbouDo paLEOJchofarecomond fZGe anZVr f zlsong teauCLClanEst'
A qHord ldLesjuire
pHalane;; Vknl,
THe pendEr mprlatolalwot
nor RFes,inaMllIVJUSlmye
AGa dheng ba inem e qHo f:d ourntRLI&LOrerks ban?reth bre seon:P
Irchumed th pxfFFbrind moind sthe the, tye aKu
Dcllullo kSoCUDWhat qSul.fow3xabre velar&AUThs mI&y
NACIIfr y me fr'dJzP, soth pth t p wigv$kstencongf a:CECOLIsQl ea ad hean, tagr; ma!
LQEno ge aKI


The mathematical trick in self-attention

In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,B = 4,8,2  #batch , time , channel